# B7 - Map back to raccoon

You've now built every piece of an AlphaZero system from scratch and seen its
training dynamics on solvable toys. Here's the explicit bridge to the real project.

## Component map

| concept | this repo (`azl/`) | raccoon |
|---|---|---|
| board/state + perspective encoding | `games/*.py` `.encode()` | `env/encoder.py`, `env/game_wrapper.py` |
| residual trunk | `foundations/blocks.py` `ConvTrunk` | conv trunk in `model/network.py` |
| policy+value network | `network.py` `AZNet` | `model/network.py` `RaccoonNet` |
| PUCT MCTS | `mcts.py` `MCTS` | `search/mcts.py` |
| chance-node sampling | `mcts.advance_through_chance` + chance branch | `_advance_through_chance` |
| self-play example gen | `selfplay.py` `play_one_game` | `train/self_play.py` |
| replay buffer | `replay_buffer.py` | `train/replay_buffer.py` |
| training loop (CE+MSE) | `trainer.py` `Coach` | `train/coach.py` |
| evaluation / arena | `evaluate.py` `play_match` | `eval/arena.py`, `eval/gnubg_harness.py` |
| supervised pretraining | `scripts/pretrain.py`, `solvers.solver_dataset` | `scripts/pretrain.py`, `data/wildbg.py` |
| ground truth for grading | `solvers.py` `ExactSolver` | GNUBG (approximate) |

## What's identical
PUCT formula, sign-flipping backup, visit-counts-as-policy-target,
`loss = CE(policy) + MSE(value)`, temperature schedule, Dirichlet noise, value
bootstrapping, chance-node sampling.

## What's simplified (and how it scales to raccoon)
- **Games are tiny and solvable** -> raccoon's backgammon is huge with no solver;
  that's why it leans on GNUBG and self-consistency.
- **No hitting/contact in DiceRace** -> real backgammon's contact is what creates the
  rich, hard position distribution (B5).
- **Serial self-play, scalar value** -> raccoon parallelises self-play and could use a
  5-way outcome head (win/gammon/backgammon).

## The intuitions worth carrying back
1. **Value head is the bottleneck.** The policy only learns because search (driven by
   value) produces better targets. (B1, B2)
2. **More simulations buy better policy targets** - often the highest-leverage knob.
   (B3, matches raccoon's exp005.)
3. **Self-play is non-stationary and can collapse** - watch visit entropy and grade on
   *held-out / off-distribution* positions, not just training loss. (B2, B3)
4. **Distributional poverty is real**: once self-play narrows, the net never sees the
   positions it's weak in. A warm start from an external teacher injects exactly those.
   (B2, B6 - matches raccoon's pretraining finding.)

In [1]:
# A concrete side-by-side: our net's config vs raccoon's defaults.
from azl.network import net_for_game
from azl.games.connect4 import Connect4
net = net_for_game(Connect4)
print("AZNet (Connect-Four):", net.config)
print()
print("raccoon RaccoonNet defaults: in_channels=17, board 2x12, channels=128,")
print("  num_blocks=6, num_actions=1352, value head tanh -> [-1, 1]")
print("Same skeleton; raccoon is just wider, deeper, and on a bigger action space.")

AZNet (Connect-Four): {'input_shape': (2, 6, 7), 'num_actions': 7, 'trunk': 'resnet', 'hidden': 128, 'channels': 32, 'num_blocks': 3}

raccoon RaccoonNet defaults: in_channels=17, board 2x12, channels=128,
  num_blocks=6, num_actions=1352, value head tanh -> [-1, 1]
Same skeleton; raccoon is just wider, deeper, and on a bigger action space.


## Where to point this next, for raccoon
- Use the **grade-on-held-out-positions** habit from here: in raccoon, keep an arena
  vs a *fixed* old checkpoint and vs GNUBG, not just training loss.
- The B3 sims-sweep predicts raccoon's biggest lever: **raise simulations** before
  adding bells and whistles.
- The B6 result argues for **external-teacher pretraining of the policy** (not just
  value) - which is exactly what moved raccoon's needle (its GNUBG-4ply distillation
  stage).